# TP 1: LDA/QDA y optimización matemática de modelos

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [1]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [2]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    #JPC cuenta la cantidad de veces que aparece cada clase

    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?
    # JPC: Porque el metodo _fit_params_ usa la variable self.log_a_priori, por lo cual tiene que estar calculada con anterioridad

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [3]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # JPC: porque y es un vector columna con el label de cada observacion. Para poder compararlo con el iterable, es necesario pasarlo a array
    # Q6: por que se usa bias=True en vez del default bias=False?
    # JPC: Porque estamos haciendo Maxima Verosimilitud, por lo cual se divide por N, no N-1
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [4]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [5]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [6]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [7]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [8]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [9]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [10]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [11]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [12]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [13]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [15]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.633102,0.223874,4.407075,1.112174,0.982407,0.018848,0.000756,0.007833,0.000678
TensorizedQDA,0.750660,0.244134,2.464528,0.683419,0.982593,0.018700,0.000715,0.012352,0.000099


In [16]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.633102,4.407075,0.982407
TensorizedQDA,0.750660,2.464528,0.982593


In [17]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.633102,0.223874,4.407075,1.112174,0.982407,0.018848,0.000756,0.007833,0.000678,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.750660,0.244134,2.464528,0.683419,0.982593,0.018700,0.000715,0.012352,0.000099,0.843395,1.788202,1.007956,0.634188


In [18]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.633102,4.407075,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.750660,2.464528,0.982593,0.843395,1.788202,1.007956,0.634188


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.
6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

## Solucion
A continuacion presentamos las respuestas a cada item propuesto

## Tensorización
### 1) Diferencias entre `QDA`y `TensorizedQDA`
**1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?**

Sobre las k clases. La observacion x es un vector de (clases, 1) unbiased_x le resta la media a cada clase, todo en una sola operacion y luego se computa la log probabilidad para todas las clases al mismo tiempo, para esa observacion

**2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`**

In [19]:
#Punto 2:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

#cantidad de clases distintas
num_classes = np.unique(y_full).size
print("Cantidad de clases: ", num_classes)


model = TensorizedQDA()
model.fit(X_full.T, y_full_encoded.T)

print('Dimensiones de x (#obs, #attrs):', X_full.shape)
print('Dimensiones de y (#obs, 1):',y_full_encoded.shape)
print(f'inv_covs en una lista de {len(model.inv_covs)} matrices de {model.inv_covs[0].shape}')
print(f'means es una lista de {len(model.means)} vectores de {model.means[0].shape}')
print('Dimensiones de tensor_inv_cov',model.tensor_inv_cov.shape)
print('Dimensiones de tensor_means',model.tensor_means.shape)

Cantidad de clases:  3
Dimensiones de x (#obs, #attrs): (178, 13)
Dimensiones de y (#obs, 1): (178, 1)
inv_covs en una lista de 3 matrices de (13, 13)
means es una lista de 3 vectores de (13, 1)
Dimensiones de tensor_inv_cov (3, 13, 13)
Dimensiones de tensor_means (3, 13, 1)


* **inv_covs** en una lista de 3 matrices de (#atributos, #atributos), o sea la matriz de covarianza de cada clase que viene de QDA luego de hacer el fit para obtener los estimadores

* **means** es una lista de 3 vectores de (#atributos, 1), o sea la lista de los mu de cada atributo  de cada clase que viene de QDA luego de hacer el fit para obtener los estimadores

* **tensor_inv_covs -> (#clases, #atributos, #atributos)** apila todas las matrices de covarianza en un solo tensor

* **tensor_means -> (#clases, #atributos, 1)** apila todos los vectores de medias

La funcion **np.stack** apila las inverzas de las matrices de conarianzas de las 3 clases en un tensor y las medias (𝑥−𝜇𝑘) en otro tensor y luego aplica la misma funcion matematica que QDA solo que en una sola operacion matricial en lugar de utilizar el for para hacer la cuenta por cada lase. La cuenta sigue siendo la misma con los mismos numeros, solo que 'paralelizada' para TensorizedQDA utilizando algebra matricial/tensorial

**Analisis del codigo**

Al metodo de calculo de log-probabilidades se le pasa 1 observacion

    _def _predict_log_conditionals(self,x):_
                                         
        _unbiased_x = x - self.tensor_means_

Esta linea hace un reshape de la muestra x (#atributos, 1) a (1, #atributos, 1) para poder relizar la operacion y queda unbiased_x (#clases, #atributos, 1)

Luego hace la resta que no es mas que quitarle la media a la muestra

En QDA teniamos: _unbiased_x =  x - self.means[class_idx]_ o sea lo mismo para una sola clase
 
     _inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x_

_unbiased_x.transpose(0,2,1)_ traspone del tensor de la muestra centrada a (#clases, 1, #atributos) para todas las clases al mismo tiempo
Luego se hace el producto tensorial (o sea con todas las clases al mismo tiempo) de la muestra centrada x inversa de covarianzas x muestra centrada segun la formula de QDA
En QDA se hace:
    _inv_cov = self.inv_covs[class_idx]_ Elije una clase de la lista de matrices de covarianzas
    _unbiased_x =  x - self.means[class_idx]_ Elije una clase de la lista de medias
    _ unbiased_x.T @ inv_cov @ unbiased_x_ _ La misma cuenta para una sola clase

Finalmente en tensorized tenemos el resultado:

    _return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()_

Finalmente _np.log(LA.det(self.tensor_inv_cov))_ es una lista de los determinantes de las inversas de las matrices de covarianza para cada clase o sea (#clases,) y _inner_prod.flatten()_ es una lista con los resultados de _ unbiased_x.T @ inv_cov @ unbiased_x_ para cada clase (la cuenta se hizo de un saque con los tensores, pero cada clase es equivalente a QDA)

Luego se escalan los dos restandos y se hace la resta de componente a componente de las dos listas y se devuelve la lista (#clases, ) de las log-probabilidades de cada clase

O sea que matematicamente, para cada clase la cuenta es la misma que en QDA

### 2) Optimización
**3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict**

Estamos intentando reemplazar el for loop de este codigo para poder predecir 'X grande' o sea 'n' observaciones al mismo tiempo basandonos en tensorizedQDA:

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

O sea que tenemos que paralelizar el calculo de Y sombrero

In [20]:
#Punto 3
class FasterQDA(TensorizedQDA):

    def _predict_log_conditionals_n(self, X):   #Ahora recibo X grande que en realidad es una matriz de n muestras
        """
        Para la nueva clase, _fit_params_ puede ser la misma que en la clase madre
        el primer lugar donde usamos las muestras es en _unbiased_x = x - self.tensor_means_
        si en tensorized x se transformaba a (1, #atributos, 1), ahora necesitamos (#clases, #atributos, n)
        de la misma forma _tensor_means__ era (#clases, #atributos, 1) y ahora tiene que ser (#clases, #atributos, n)
        """

        #Veo que forma tiene X (no se cuantas muestras tiene)   
        _, n = X.shape     
        #Consigo los numeros de clases y atributos
        k, a, _ = self.tensor_means.shape

        #Formateo X
        X_format = np.stack([X] * k, axis=0)

        #Formateo self.tensor_means
        means_format = np.stack([self.tensor_means[:,:,0]] * n, axis=2)

        #Ahora  el tensor de las muestras menos la media 
        unbiased = X_format - means_format

        #Matrices de interacciones por clase de n x n
        inter_m = unbiased.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased

        #La diagonal, como dice el enunciado da las auto-interacciones de (#clases, n)
        auto_interac = np.diagonal(inter_m, axis1=1, axis2=2)

        return 0.5*np.log(LA.det(self.tensor_inv_cov))[:, np.newaxis] - 0.5 * auto_interac

    def predict(self, X):

        log_posteriori = self.log_a_priori[:, np.newaxis] + self._predict_log_conditionals_n(X)
        y_hat = np.argmax(log_posteriori, axis=0)
        return y_hat.reshape(1, -1)

In [21]:

model_tqda = TensorizedQDA()
model_tqda.fit(X_full.T, y_full_encoded.T)
mis_X = X_full.T

mis_y_tqda = np.array([model_tqda._predict_one(mis_X[:, i].reshape(-1, 1))
    for i in range(mis_X.shape[1])
])
print(mis_y_tqda.shape)
print(mis_y_tqda)


(178,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [22]:
model_fqda = FasterQDA()
model_fqda.fit(X_full.T, y_full_encoded.T)
mis_y_fqda = model_fqda.predict(mis_X)
print(mis_y_fqda.shape)
print(mis_y_fqda)

(1, 178)
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1
  1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
  1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2
  2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]]


Milagrosamente predicen lo mismo.

**4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.**

En el codigo de arriba en:

    _inter_m = unbiased.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased_

En realidad es un tensor con de todas las matrices de n x n de cada clase

**5.Demostración**

In [23]:
np.random.seed(88)
rand1 = np.random.randn(178, 13)
rand2 = np.random.randn(13, 178)

prod = rand1 @ rand2
diag_prod = np.diag(prod)

diag_nuevo_metodo = np.sum(rand1 * rand2.T, axis=1)
#print(diag_prod)
#print(diag_nuevo_metodo)

#Son iguales?
print(np.allclose(diag_prod, diag_nuevo_metodo))

True


**6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`**

In [24]:
class EfficientQDA(TensorizedQDA):

    def _predict_log_conditionals_n(self, X):   #Ahora recibo X grande que en realidad es una matriz de n muestras
        """"
        Para la nueva clase, _fit_params_ puede ser la misma que en la clase madre
        el primer lugar donde usamos las muestras es en _unbiased_x = x - self.tensor_means_
        si en tensorized x se transformaba a (1, #atributos, 1), ahora necesitamos (#clases, #atributos, n)
        de la misma forma _tensor_means__ era (#clases, #atributos, 1) y ahora tiene que ser (#clases, #atributos, n)
        """
        #Veo que forma tiene X (no se cuantas muestras tiene)   
        _, n = X.shape     
        #Consigo los numeros de clases y atributos
        k, a, _ = self.tensor_means.shape

        #Formateo X
        X_format = np.stack([X] * k, axis=0)

        #Formateo self.tensor_means
        means_format = np.stack([self.tensor_means[:,:,0]] * n, axis=2)

        #Ahora  el tensor de las muestras menos la media 
        unbiased = X_format - means_format

        #Ahora el producto tensorial del nuevo metodo propuesto
        #Antes teniamos diagonal(unbiased.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased)
        #Separo el producto en unbiased.transpose(0, 2, 1) y (self.tensor_inv_cov @ unbiased)
        #A -> unbiased
        #B_T -> self.tensor_inv_cov @ unbiased
        prod_segundo = self.tensor_inv_cov @ unbiased
        auto_interac = np.sum(unbiased * prod_segundo)

        return 0.5*np.log(LA.det(self.tensor_inv_cov))[:, np.newaxis] - 0.5 * auto_interac

    def predict(self, X):

        log_posteriori = self.log_a_priori[:, np.newaxis] + self._predict_log_conditionals_n(X)
        y_hat = np.argmax(log_posteriori, axis=0)
        return y_hat.reshape(1, -1)

In [25]:
model_eqda = EfficientQDA()
model_eqda.fit(X_full.T, y_full_encoded.T)
mis_y_eqda = model_fqda.predict(mis_X)
print(mis_y_eqda.shape)
print(mis_y_eqda)
print(np.allclose(mis_y_fqda, mis_y_eqda))


(1, 178)
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1
  1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
  1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2
  2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]]
True


FasterQDA y EfficientQDA Predicen las mismas clases

**7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?**

In [26]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [27]:
# bencheamos un par
to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b.bench(model)

b.summary()

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.970523,0.612335,7.717296,1.887667,0.982407,0.018700,0.000680,0.007789,0.000295
TensorizedQDA,1.001133,0.281235,3.708566,0.905625,0.982593,0.018700,0.000673,0.012352,0.000103
FasterQDA,1.125517,0.451083,0.411022,0.099038,0.985741,0.018105,0.000934,0.142174,0.000093
EfficientQDA,1.046086,0.275827,0.337350,0.088244,0.312963,0.018228,0.000700,0.091873,0.000146


Se observa una gran diferencia en los parametros de test que es el punto de crear distintas variantes de QDA 

* **train_median_ms:** Deberian ser iguales porque usan el mismo metodo. EfficientQDA a veces me da mucho mas alto con mucha mas train_std_ms, sera por los procesos que ocupaban al sistema en ese momento?

* **test_median_ms:** Tiene sentido. QDA calcula por clase y luego el for loop para las observaciones, TensorizedQDA paraleliza las clases y luego el for loop para las observaciones, FasterQDA resulta mas rapido que TensorizedQDA (lo cual estaba en duda) ya que procesa todas las observaciones de una pasada y EfficientQDA es apenas mas lento que FasterQDA

* **mean_accuracy:** EfficientQDA da muy mal, pero cuando lo probamos de manera aislada predecia igual que los demas

* **test_mem_median_mb:** Tiene mucho sentido. QDA hace las cuentas secuencialmente por lo cual necesita menos memoria, TensorizedQDA paraleliza, por lo cual tiene que armar el array con todas las inverzas de las covarianzas para todas las clases etc por lo cual usa mas memoria, FasterQDA es similar a TensorizedQDA pero arma lo mismo para todas las 178 muestras de una vez y EfficientQDA no tiene que armar  

## Cholesky
### 3) Diferencias entre implementaciones de `QDA_Chol`
**8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?**



$(AB)^{-1} = B^{-1}A^{-1}$

$A^{-1} = (LL^{T})^{-1} = (L^{T})^{-1}L^{-1}$

Segun esto, podriamos invertir las matrices de covarianza como se escribe arriba por medio del producto de las inversas de los factores Cholesky que no requieren computar la inversion sino resolver

$LX = I$

por medio de la funcion solve_triangular() donde $X = L^{-1}$

La ventaja principal es que es el método numéricamente más estable de todos, evitando por completo los errores de redondeo acumulados por inversiones explícitas.
Yo solo mencionaria el tema de la velocidad mas que estabilidad numerica por no tener que calcular las inversas.

**9. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones**

En el metodo _fit_params: 
* **QDA_Chol1** calcula las inversas de L (factorizaciones de Cholesky) de las matrices de covarianza de cada atributo
* **QDA** calcula las inversas de las matrices de covarianza de cada atributo en si
* Las medias son calculadas de la misma forma en ambas clases

En el metodo _predict_log_conditional
* La muestra centrada es calculada de la misma forma en ambos 
* QDA_Chol1 utiliza:

$(x-\mu)^{T}\Sigma^{-1}(x-\mu)$

$(x-\mu)^{T}(L^{T})^{-1}L^{-1}(x-\mu)$

el codigo hace un cambio de variables

$y = L^{-1}(x-\mu)$

que lo resuelve como:

_y = L_inv @ unbiased_x_

por otro lado

$y^{T} = (x-\mu)^{T}(L^{T})^{-1}$

$y^{T}y = ||y||^{2} = (x-\mu)^{T}(L^{T})^{-1}L^{-1}(x-\mu) = (x-\mu)^{T}\Sigma^{-1}(x-\mu)$

y

$det(\Sigma^{-1}) = det((L^{T})^{-1}L^{-1}) = det((L^{T})^{-1}) det(L^{-1})$

$det(L^{T})^{-1} = 1/det(L^{T}) = 1/det(L)$

$det(\Sigma^{-1}) = det(L)^{-2}$

como L es triangular, el determinante se puede calcular como el producto de todos los elementos de la diagonal _L_inv.diagonal().prod()_ y tomando el logaritmo de ambos lados el cuadrado baja multiplicando x 2 que se cancela con el 0.5 de la expresion. Queda que el log del determinante de la matriz de covarianzas se puede calcular como np.log(L_inv.diagonal().prod()) (sin el menos porque teniamos el menos dentro del determinante de la matriz de covarianza y sin el 0.5 porque se cancela con el 2 que baja multiplicando del cuadrado al toma el logaritmo)

Con lo cual:

_return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()_

y

return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

retornan lo mismo

**10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?** 

Las tres variantes utilizan la Descomposición de Cholesky ($\Sigma = L L^T$) para sustituir inversión explícita de la matriz de covarianza por la manipulación de una matriz triangular inferior ($L$), pero difieren en el momento y la estrategia que usan para resolver el álgebra.

* **QDA_Chol1:** En el entrenamiento calcula la matriz de Cholesky $L$ para cada clase y luego calcula su inversa explícita $L^{-1}$ usando el método general de NumPy (LA.inv(L)). Por otro lado, en la predicción multiplica directamente el vector de datos centrados por la inversa precalculada (y = L_inv @ unbiased_x). 

* **QDA_Chol2:** En el entrenamiento, es el más rápido de los tres en esta etapa. Solo calcula y almacena la matriz triangular de Cholesky $L$. No realiza ninguna inversión matricial, reduciendo el costo computacional en el entrenamiento, mientras que en la predicción en lugar de hacer una multiplicación, debe resolver el sistema lineal triangular $L y = x - \mu$ para cada muestra usando solve_triangular. La ventaja principal es que es el método numéricamente más estable de todos, evitando por completo los errores de redondeo acumulados por inversiones explícitas.

* **QDA_Chol3:** Al igual que el Chol1, invierte explícitamente la matriz $L$, pero no usa una función genérica. Utiliza dtrtri de LAPACK, una rutina de muy bajo nivel diseñada específicamente para invertir matrices triangulares de forma óptima. En la predicción es idéntico a QDA_Chol1, ejecutando una veloz multiplicación de matriz por vector con la inversa triangular optimizada.

In [28]:
# Windows - me tira numpy 2.4.3 y scipy 1.17.1
!python --version
# !pip freeze | findstr /R "numpy scipy"
!pip freeze | grep -E "numpy|scipy" #linux y mac, para windows usar el comando comentado de arriba

Python 3.12.13
numpy @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_numpy_1773839284/work/dist/numpy-2.4.3-cp312-cp312-linux_x86_64.whl#sha256=bd15e3dc0b3b571b30f62a64a69d71a6d27674e5d0ed5436cde406f2704c93e9
scipy @ file:///home/conda/feedstock_root/build_artifacts/scipy-split_1771879143114/work/dist/scipy-1.17.1-cp312-cp312-linux_x86_64.whl#sha256=141f721c574c72812b8184918c7a44890af730e8cea9be70bb2be8428a2b6f25


**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [29]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


**11. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?**

In [30]:
to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    b.bench(model)

b.summary()

QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

...

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,14.021584,2.488102,2653.434088,115.047875,0.886117,0.269287,0.001992,0.097432,0.000751
TensorizedQDA,13.051694,2.082129,580.110692,28.600256,0.885303,0.268700,0.002143,0.154343,0.000206
FasterQDA,15.756874,3.536783,3882.063633,216.848037,0.884827,0.269188,0.001919,3224.727264,0.000719
EfficientQDA,19.686073,6.079312,65.027231,18.432644,0.037840,0.269676,0.002171,63.594063,0.000026
QDA_Chol1,16.112323,2.228025,1350.372752,63.430502,0.884770,0.269087,0.002137,0.094976,0.000423
QDA_Chol2,14.260125,2.720288,4828.290211,146.249329,0.885433,0.269094,0.001999,0.096702,0.000583
QDA_Chol3,14.464753,2.237743,1305.913236,78.922903,0.885807,0.268999,0.001876,0.094740,0.000328


* Todas las variantes mantienen exactamente la misma precisión media, lo que demuestra que las modificaciones de Cholesky y de vectorización son matemáticamente equivalentes al QDA clásico.
* Entre las descomposiciones de Cholesky:
QDA_Chol3: mejor opción global. Tiene la mayor accuracy media: 0.885807, presenta uno de los menores tiempos de inferencia/prueba: 779.94 ms, tiene la menor variabilidad en el tiempo de prueba (46.05 ms) entre las tres variantes y por ultimo el consumo de memoria es prácticamente idéntico al resto (~0.269 MB en entrenamiento y ~0.095 MB en prueba).
QDA_Chol1: segunda mejor, muy cercana a Chol3.
QDA_Chol2: peor alternativa debido a su elevado tiempo de ejecución aunque su accuracy (0.885433) es similar a las otras dos.

Además, comparando con todos los métodos de la figura, las variantes QDA_Chol mantienen una precisión similar a QDA estándar (~88.5%), pero ninguna supera a EfficientQDA en velocidad; sin embargo, este último parece tener un problema importante de accuracy (≈3.8%), por lo que probablemente existe algún error o trade-off muy agresivo en la implementación.

* En términos globales, al buscar el balance entre precisión de QDA y reducción del coste computacional, TensorizedQDA es la alternativa más equilibrada. Entre las variantes basadas en Cholesky, QDA_Chol3 es la más recomendable. Por el contrario, QDA_Chol2 y especialmente EfficientQDA muestran las peores relaciones entre rendimiento predictivo y eficiencia computacional.

* El caso de EfficientQDA es especial, dado que obtiene con diferencia los mejores tiempos de entrenamiento y prueba, su accuracy cae hasta aproximadamente 3.8 %, muy por debajo del resto de modelos. La optimización implementada compromete la capacidad predictiva del clasificador.

**12. Implementación de TensorizedChol**

In [31]:
class TensorizedChol(QDA_Chol3):
    def _fit_params(self, X, y):
        # Ejecutamos el fit original de QDA_Chol3 para obtener medias y L_invs
        super()._fit_params(X, y)
        
        # 2. Tensorizamos los parámetros empaquetándolos
        self.means_tensor = np.stack(self.means, axis=0)      # (#clases, #obs,  #obs )
        self.L_inv_tensor = np.stack(self.L_invs, axis=0)     # (#clases, #obs,  #obs )
        
        # Calculamos las log-determinantes a partir de las L_invs generadas por el padre
        log_dets_list = [np.log(L_inv.diagonal().prod()) for L_inv in self.L_invs]
        self.log_det_tensor = np.array(log_dets_list)[:, np.newaxis] # Formato: (K, 1)

    def predict(self, X):
        # X viene con dimensiones (P_features, N_observaciones)
        X_tensor = X[np.newaxis, :, :] # Formato: (1, P_features, N_observaciones)
        
        # Restamos las medias de todas las clases en paralelo
        diff = X_tensor - self.means_tensor # Formato: (K_clases, P_features, N_observaciones)
        
        # Multiplicación matricial por lotes
        y = self.L_inv_tensor @ diff
        
        # Término cuadrático
        quad_term = np.sum(y**2, axis=1) # Formato: (K_clases, N_observaciones)
        
        # Probabilidad a posteriori
        log_priors = np.array(self.log_a_priori)[:, np.newaxis] # (K_clases, 1)
        
        # Nota el signo + antes del log_det_tensor porque estamos usando L_inv en lugar de L
        log_posterior = log_priors + self.log_det_tensor - 0.5 * quad_term
        
        return np.argmax(log_posterior, axis=0)

**13. Implementación de EfficientChol**

In [32]:
class EfficientChol(QDA_Chol3):
    def predict(self, X):
        N_samples = X.shape[1]
        K_clases = len(self.log_a_priori)
        
        log_posterior = np.zeros((K_clases, N_samples))
        
        for j in range(K_clases):
            diff = X - self.means[j]
            y = self.L_invs[j] @ diff
            quad_term = np.sum(y**2, axis=0)
            
            # Calculamos individualmente el término del determinante usando L_inv
            log_det_j = np.log(self.L_invs[j].diagonal().prod())
            log_posterior[j, :] = self.log_a_priori[j] + log_det_j - 0.5 * quad_term
            
        return np.argmax(log_posterior, axis=0)

In [33]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


In [34]:
to_bench = [TensorizedChol, EfficientChol]

for model in to_bench:
    b.bench(model)

b.summary()

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
TensorizedChol,13.777912,3.735771,22.573315,5.494431,0.886117,0.269538,0.002182,38.997036,0.000580
EfficientChol,14.546816,2.950402,12.086421,2.603218,0.885303,0.268750,0.002200,2.686666,0.000237


Los resultados muestran lo siguiente:

1. Las variantes algebraicamente equivalentes conservan la accuracy.
2. Cholesky reduce significativamente el tiempo de cálculo.
3. La vectorización aporta las mayores ganancias de velocidad. Pero, a mayor velocidad tambien es mayor el consumo de memoria.
4. Algunas optimizaciones teóricas (como FasterQDA) pueden no ser beneficiosas en la práctica debido al overhead de implementación.
5. TensorizedChol y EfficientChol logran acelerar QDA sin degradar la precisión, mostrando que la optimización basada en Cholesky y vectorización es la estrategia más efectiva entre todas las variantes evaluadas.

Con respecto a la evaluación de las 9 lo resumimos en la siguiente tabla:


| Modelo         | Accuracy | Test (ms) | Comentario                           |
| -------------- | -------- | --------- | ------------------------------------ |
| QDA            | 0.8861   | 1590.5    | Referencia base                      |
| TensorizedQDA  | 0.8853   | 357.3     | Misma precisión con gran aceleración |
| FasterQDA      | 0.8848   | 2983.0    | Más lento que QDA                    |
| EfficientQDA   | 0.0378   | 38.8      | Accuracy inaceptable                 |
| QDA_Chol1      | 0.8848   | 794.8     | Buena alternativa                    |
| QDA_Chol2      | 0.8854   | 2786.4    | Muy lento                            |
| QDA_Chol3      | 0.8858   | 779.9     | Mejor variante Chol                  |
| TensorizedChol | 0.8861   | 24.1      | Excelente balance                    |
| EfficientChol  | 0.8853   | 8.0       | Mejor rendimiento global             |
